<a href="https://colab.research.google.com/github/tylerfuentes/ML_Class_LORA/blob/main/notebooks/colab_a100_unsloth_qwen_finance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
%%bash
# Install GitHub CLI
curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | sudo dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | sudo tee /etc/apt/sources.list.d/github-cli.list > /dev/null
sudo apt-get update -qq && sudo apt-get install -y gh
gh --version


Reading package lists...
Building dependency tree...
Reading state information...
The following packages will be upgraded:
  gh
1 upgraded, 0 newly installed, 0 to remove and 98 not upgraded.
Need to get 14.7 MB of archives.
After this operation, 826 kB of additional disk space will be used.
Get:1 https://cli.github.com/packages stable/main amd64 gh amd64 2.95.0 [14.7 MB]
Fetched 14.7 MB in 0s (86.2 MB/s)
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../archives/gh_2.95.0_amd64.deb ...
Unpacking gh (2.95.0) over (2.93.0) ...
Setting up gh (2.95.0) ...
Processing triggers for man-db (2.10.2-1) ...
gh version 2.95.0 (2026-06-17)
https://github.com/cli/cli/releases/tag/v2.95.0


8+1 records in
8+1 records out
4528 bytes (4.5 kB, 4.4 KiB) copied, 0.0536327 s, 84.4 kB/s
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [ ]:
import os, subprocess
from google.colab import userdata

# Load token from Colab Secrets (add GITHUB_TOKEN in the key icon on the left panel)
token = userdata.get('GITHUB_TOKEN')
proc = subprocess.run(
    ['gh', 'auth', 'login', '--hostname', 'github.com', '--git-protocol', 'https', '--with-token'],
    input=token, text=True, capture_output=True
)
print(proc.stdout or proc.stderr)

subprocess.run(['git', 'config', 'user.email', 'tfilly93@gmail.com'], check=True)
subprocess.run(['git', 'config', 'user.name', 'tfilly'], check=True)
subprocess.run(['gh', 'auth', 'status'], check=True)


# Colab A100 Unsloth Qwen Finance

Thin notebook for VS Code + Google Colab execution. Reusable logic lives in repo scripts under `scripts/colab/`.

In [8]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get('ML_CLASS_LORA_REPO', 'https://github.com/nathanaelguitar/ML_Class_LORA.git')
REPO_BRANCH = os.environ.get('ML_CLASS_LORA_BRANCH', 'main')
REPO_DIR = Path('/content/ML_Class_LORA')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
# FinGPT submodule must be re-inited after every VM reset
subprocess.run(['git', '-C', str(REPO_DIR), 'submodule', 'update', '--init', '--recursive'], check=True)
os.chdir(REPO_DIR)
print('repo_dir', REPO_DIR)


repo_dir /content/ML_Class_LORA


In [5]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!python scripts/colab/verify_runtime.py --require-gpu-substring A100 --min-memory-gb 35


NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07
gpu_name: NVIDIA A100-SXM4-80GB
gpu_memory_gb: 79.25
torch_version: 2.11.0+cu128
torch_cuda_version: 12.8
cuda_bf16_supported: True
unsloth_version: <not installed>


In [6]:
!python scripts/colab/bootstrap_unsloth_env.py


Using PyTorch wheel index for cu128
+ /usr/bin/python3 -m pip install -q --upgrade pip setuptools wheel
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
+ /usr/bin/python3 -m pip install -q --force-reinstall --index-url https://download.pytorch.org/whl/cu128 torch==2.10.0 torchvision==0.25.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
cuda-python 12.9.7 requires cuda-bi

In [7]:
import subprocess, sys

# Run the failing install step verbosely to expose the actual error
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--upgrade", "--force-reinstall",
        "pyyaml",
        "unsloth==2026.6.7",
        "transformers==5.5.0",
        "datasets==4.3.0",
        "trl==0.24.0",
        "peft==0.19.1",
        "bitsandbytes==0.49.2",
        "accelerate==1.13.0",
        "huggingface_hub==1.14.0",
    ],
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")
print("Return code:", result.returncode)


STDOUT: sfully uninstalled diffusers-0.38.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2026.6.7
    Uninstalling unsloth_zoo-2026.6.7:
      Successfully uninstalled

In [10]:
import importlib.metadata
import torch

print('torch', torch.__version__)
print('cuda', torch.version.cuda)
print('bf16', torch.cuda.is_bf16_supported())
print('unsloth', importlib.metadata.version('unsloth'))


torch 2.10.0+cu128
cuda 12.8
bf16 True
unsloth 2026.6.7


In [27]:
from pathlib import Path
import yaml

CONFIG_PATH = Path('config/colab_paths.local.yaml')
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path('config/colab_paths.example.yaml')
    print('WARNING: colab_paths.local.yaml not found, falling back to example config')
cfg = yaml.safe_load(CONFIG_PATH.read_text())
for key in ('train_file', 'train_eval_file', 'val_file', 'test_file'):
    path = Path(cfg['datasets'][key])
    size_mb = round(path.stat().st_size / (1024 ** 2), 2) if path.exists() else None
    print(key, path, 'exists=', path.exists(), 'size_mb=', size_mb)


train_file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/train.jsonl exists= True size_mb= 1747.52
train_eval_file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/train_eval.jsonl exists= True size_mb= 3.33
val_file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/val.jsonl exists= True size_mb= 96.91
test_file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/test.jsonl exists= True size_mb= 96.71


In [25]:
# Write a local config override with the correct Drive folder paths.
# Writes to colab_paths.local.yaml (gitignored) so the example file stays clean.
import yaml, subprocess
from pathlib import Path

TARGET_FOLDER_ID = '1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga'
DRIVE_ROOT_BY_ID = f'/content/drive/.shortcut-targets-by-id/{TARGET_FOLDER_ID}/ML_Class_LORA'

EXAMPLE_PATH = Path('config/colab_paths.example.yaml')
LOCAL_PATH = Path('config/colab_paths.local.yaml')
OLD_ROOT = '/content/drive/MyDrive/ML_Class_LORA'

cfg = yaml.safe_load(EXAMPLE_PATH.read_text())
for key in list(cfg['drive'].keys()):
    cfg['drive'][key] = cfg['drive'][key].replace(OLD_ROOT, DRIVE_ROOT_BY_ID)
for key in list(cfg['datasets'].keys()):
    cfg['datasets'][key] = cfg['datasets'][key].replace(OLD_ROOT, DRIVE_ROOT_BY_ID)

LOCAL_PATH.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))

# Ensure it's gitignored
gitignore = Path('.gitignore')
if gitignore.exists() and 'colab_paths.local.yaml' not in gitignore.read_text():
    with gitignore.open('a') as f:
        f.write('\nconfig/colab_paths.local.yaml\n')

print('Local config written to:', LOCAL_PATH)
print('Drive root →', cfg['drive']['root'])
for k, v in cfg['drive'].items():
    print(f'  {k}: {v}')


Local config written to: config/colab_paths.local.yaml
Drive root → /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA
  root: /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA
  data_gold: /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold
  adapters: /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/adapters
  checkpoints: /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/checkpoints
  outputs: /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/outputs
  manifests: /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/manifests


In [ ]:
from huggingface_hub import snapshot_download
import yaml

cfg = yaml.safe_load(CONFIG_PATH.read_text())
snapshot_download(repo_id=cfg['model']['model_id'], local_files_only=cfg['model'].get('local_files_only', False))


In [14]:
import os

if os.environ.get('HF_TOKEN'):
    get_ipython().system('python scripts/colab/check_hf_auth.py --token-env HF_TOKEN')
else:
    print('HF_TOKEN not set; skipping Hugging Face auth check.')


HF_TOKEN not set; skipping Hugging Face auth check.


In [16]:
from pathlib import Path
root = Path(cfg['drive']['root'])
print('root exists:', root.exists())
if root.exists():
    print('root contents:', sorted(p.name for p in root.iterdir())[:20])
else:
    # Try the MyDrive path as fallback diagnostic
    fallback = Path('/content/drive/MyDrive')
    print('MyDrive exists:', fallback.exists())
    print('MyDrive contents:', sorted(p.name for p in fallback.iterdir())[:20])


root exists: True
root contents: ['ML_Class_LORA']


In [18]:
from pathlib import Path

for key in ('adapters', 'checkpoints', 'outputs', 'manifests'):
    path = Path(cfg['drive'][key])
    print(key, path, 'exists=', path.exists())
    if path.exists():
        children = sorted(item.name for item in path.iterdir())[:10]
        print('  first_entries=', children)


adapters /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/adapters exists= True
  first_entries= ['qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z-sanity']
checkpoints /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/checkpoints exists= True
  first_entries= ['qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z', 'qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z-checkpoint-4500-backup-keydrift', 'qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z-sanity']
outputs /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/outputs exists= True
  first_entries= ['evals', 'sanity_logs', 'train_logs']
manifests /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/manifests exists= True
  first_entries= ['qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z-sanity_manifest.json']


In [36]:
RUN_NAME = cfg['training']['run_name']
SANITY_STEPS = 5
get_ipython().system(f'python scripts/colab/sanity_gate.py --config config/colab_paths.local.yaml --run-name {RUN_NAME} --sanity-steps {SANITY_STEPS}')


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Running sanity train: /usr/bin/python3 /content/ML_Class_LORA/training/train_finance_lora_unsloth.py --model-id Qwen/Qwen3.6-27B --train-file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/train.jsonl --eval-file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/train_eval.jsonl --test-file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/test.jsonl --output-dir /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/checkpoints/qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z-sanity --epochs 1.0 --lr 0.0002 --per-device-train-batch-size 4 --gradient-accumulation-steps 4 --save-steps 5 --logging-steps 1 --eval-steps 5 --lora-r 16 --lora-alpha 32 --lora-dropout 0.0 --max-seq-length 102

In [35]:
import subprocess
result = subprocess.run(
    ['git', 'submodule', 'update', '--init', '--recursive'],
    cwd='/content/ML_Class_LORA', capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print('returncode:', result.returncode)


Submodule path 'external/FinGPT': checked out '484a81d437977d1cf8d11dcf6093794571eb6e11'

Submodule 'external/FinGPT' (https://github.com/AI4Finance-Foundation/FinGPT) registered for path 'external/FinGPT'
Cloning into '/content/ML_Class_LORA/external/FinGPT'...

returncode: 0


In [37]:
import os
from pathlib import Path
log = Path(cfg['drive']['outputs']) / 'sanity_logs' / f'{cfg["training"]["run_name"]}-sanity.log'
print('exists:', log.exists())
if log.exists():
    print('size:', log.stat().st_size, 'bytes')
    print(log.read_text()[-3000:])
else:
    print('(not yet created)')


exists: False
(not yet created)


In [38]:
print("alive")


alive


In [ ]:
import json
from pathlib import Path

manifest_path = Path(cfg['drive']['manifests']) / f"{RUN_NAME}-sanity_manifest.json"
payload = json.loads(manifest_path.read_text())
print('manifest_path', manifest_path)
for row in payload['generations']:
    print('example', row['example_index'])
    print('generated_output', row['generated_output'][:800])
    print('---')


In [ ]:
# Launch this only after the sanity gate succeeds.
# --resume-latest picks up from the latest checkpoint in the run dir.
# --max-steps-override is ABSOLUTE (not a delta). Chunks: 12500→15500→18500→21500→24500→27500→30500→32750
RUN_NAME = cfg['training']['run_name']
MAX_STEPS = 12500
get_ipython().system(f'python scripts/colab/train_unsloth_from_config.py --config config/colab_paths.local.yaml --run-name {RUN_NAME} --resume-latest --max-steps-override {MAX_STEPS}')


Running: /usr/bin/python3 /content/ML_Class_LORA/training/train_finance_lora_unsloth.py --model-id Qwen/Qwen3.6-27B --train-file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/train.jsonl --eval-file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/train_eval.jsonl --test-file /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/data/gold/test.jsonl --output-dir /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/checkpoints/qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z --epochs 1.0 --lr 0.0002 --per-device-train-batch-size 4 --gradient-accumulation-steps 4 --save-steps 500 --logging-steps 25 --eval-steps 500 --lora-r 16 --lora-alpha 32 --lora-dropout 0.0 --max-seq-length 1024 --gpu-memory-utilization 0.95 --max-total-examples 0 --resume-from-checkpoint /content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55

In [ ]:
from pathlib import Path

log = Path(cfg['drive']['outputs']) / 'train_logs' / f'{cfg["training"]["run_name"]}.log'
ckpt_dir = Path(cfg['drive']['checkpoints']) / cfg['training']['run_name']

print('=== Latest checkpoints ===')
if ckpt_dir.exists():
    ckpts = sorted([p.name for p in ckpt_dir.iterdir() if p.name.startswith('checkpoint-')],
                   key=lambda x: int(x.split('-')[1]))
    for c in ckpts[-5:]:
        print(' ', c)

print('\n=== Last 50 lines of training log ===')
if log.exists():
    lines = log.read_text().splitlines()
    print('\n'.join(lines[-50:]))
else:
    print('(log not found)')


In [ ]:
get_ipython().system(f'python scripts/colab/run_post_training_evals.py --config config/colab_paths.local.yaml --run-name {RUN_NAME}')


In [ ]:
import subprocess
from pathlib import Path

# Find latest checkpoint for commit message
ckpt_dir = Path(cfg['drive']['checkpoints']) / RUN_NAME
checkpoints = sorted(
    [p.name for p in ckpt_dir.iterdir() if p.name.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[1])
) if ckpt_dir.exists() else []
latest = checkpoints[-1] if checkpoints else 'unknown'

def git(args, **kw):
    return subprocess.run(['git'] + args, cwd='/content/ML_Class_LORA', check=True, **kw)

git(['add', '.gitignore'])
git(['add', '-u'])  # stages modifications to tracked files; colab_paths.local.yaml is gitignored

status = subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd='/content/ML_Class_LORA')
if status.returncode == 0:
    print('Nothing to commit.')
else:
    git(['commit', '-m', f'training: chunk complete at {latest} for {RUN_NAME}'])
    subprocess.run(['gh', 'repo', 'set-default', 'tylerfuentes/ML_Class_LORA'], cwd='/content/ML_Class_LORA')
    git(['push', 'origin', 'main'])
    print(f'Pushed: {latest}')


In [ ]:
import os
import subprocess

ENABLE_HF_UPLOAD = os.environ.get('ENABLE_HF_UPLOAD', '').lower() in {'1', 'true', 'yes'}
if ENABLE_HF_UPLOAD:
    subprocess.run([
        'python',
        'scripts/colab/push_final_adapter.py',
        '--config',
        'config/colab_paths.example.yaml',
        '--run-name',
        RUN_NAME,
        '--enable-upload',
    ], check=True)
else:
    print('Skipping Hugging Face upload; set ENABLE_HF_UPLOAD=1 and HF_TOKEN to enable private adapter push.')


In [ ]:
import os, time
from pathlib import Path

ckpt_dir = Path('/content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/checkpoints/qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z')
log_path = Path('/content/drive/.shortcut-targets-by-id/1-I2fhEUE2NE9RTYpqn55jXLDsCZqw0ga/ML_Class_LORA/outputs/train_logs/qwen36-27b-wrds-500k-unsloth-gb10-rerun-20260616T2330Z.log')

print(f'Check time: {time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())}')
print()

if ckpt_dir.exists():
    ckpts = sorted([p.name for p in ckpt_dir.iterdir() if p.name.startswith('checkpoint-')],
                   key=lambda x: int(x.split('-')[1]))
    print(f'Checkpoints ({len(ckpts)} total): {ckpts[-5:]}')
else:
    print('checkpoint dir missing')

print()
if log_path.exists():
    lines = log_path.read_text().splitlines()
    print(f'Log: {len(lines)} lines, last modified {time.strftime("%H:%M:%S UTC", time.gmtime(log_path.stat().st_mtime))}')
    print('\n'.join(lines[-20:]))
else:
    print('log not found')


In [ ]:
print("kernel_alive_check")

In [ ]:
print("alive2")

In [ ]:
print("alive3")

In [ ]:
print("alive4")

In [ ]:
print("alive5")

In [ ]:
print("alive6")

In [ ]:
print("alive7")

In [ ]:
import time; print(f"alive8 at {time.strftime('%H:%M:%S UTC', time.gmtime())}")

In [ ]:
print("alive9")

In [ ]:
print("alive10")

In [ ]:
print("alive11")